In [1]:
import pandas as pd
import sqlite3

# 1. Connect to SQLite database
conn = sqlite3.connect("retail_sales.db")
cursor = conn.cursor()

# 2. Create Tables
cursor.executescript("""
DROP TABLE IF EXISTS Sales;
DROP TABLE IF EXISTS Stores;

CREATE TABLE Stores (
    StoreID INTEGER PRIMARY KEY,
    StoreName TEXT,
    Region TEXT
);

CREATE TABLE Sales (
    TransactionID INTEGER PRIMARY KEY,
    StoreID INTEGER,
    Category TEXT,
    Amount REAL,
    TransactionDate TEXT,
    FOREIGN KEY (StoreID) REFERENCES Stores(StoreID)
);

INSERT INTO Stores VALUES (1, 'Birmingham Central', 'Midlands');
INSERT INTO Stores VALUES (2, 'London West', 'South');
INSERT INTO Stores VALUES (3, 'Manchester North', 'North');

INSERT INTO Sales VALUES (101, 1, 'Electronics', 450.00, '2024-01-15');
INSERT INTO Sales VALUES (102, 1, 'Clothing', 120.50, '2024-01-16');
INSERT INTO Sales VALUES (103, 2, 'Electronics', 890.00, '2024-01-18');
INSERT INTO Sales VALUES (104, 2, 'Beauty', 75.00, '2024-01-20');
INSERT INTO Sales VALUES (105, 3, 'Electronics', 620.00, '2024-01-22');
INSERT INTO Sales VALUES (106, 3, 'Clothing', 210.00, '2024-01-25');
INSERT INTO Sales VALUES (107, 1, 'Beauty', 130.00, '2024-01-28');
INSERT INTO Sales VALUES (108, 2, 'Clothing', 340.00, '2024-02-01');
""")

# 3. SQL Query: Store & Regional Sales Aggregation with JOIN
query = """
SELECT
    st.Region,
    st.StoreName,
    s.Category,
    COUNT(s.TransactionID) AS Total_Transactions,
    SUM(s.Amount) AS Total_Revenue,
    ROUND(AVG(s.Amount), 2) AS Avg_Transaction_Value
FROM Sales s
JOIN Stores st ON s.StoreID = st.StoreID
GROUP BY st.Region, st.StoreName, s.Category
ORDER BY Total_Revenue DESC;
"""

df_results = pd.read_sql_query(query, conn)
print("--- SQL Aggregation Query Results ---")
print(df_results)

# Close connection
conn.close()


--- SQL Aggregation Query Results ---
     Region           StoreName     Category  Total_Transactions  \
0     South         London West  Electronics                   1   
1     North    Manchester North  Electronics                   1   
2  Midlands  Birmingham Central  Electronics                   1   
3     South         London West     Clothing                   1   
4     North    Manchester North     Clothing                   1   
5  Midlands  Birmingham Central       Beauty                   1   
6  Midlands  Birmingham Central     Clothing                   1   
7     South         London West       Beauty                   1   

   Total_Revenue  Avg_Transaction_Value  
0          890.0                  890.0  
1          620.0                  620.0  
2          450.0                  450.0  
3          340.0                  340.0  
4          210.0                  210.0  
5          130.0                  130.0  
6          120.5                  120.5  
7           7